In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

def fetch_pullpush_data(query, subreddit, total_target=2000):
    """
    使用 PullPush API 批次爬取 Reddit 貼文，並包含自動跳轉時間機制
    """
    all_data = []
    before_time = int(time.time())

    print(f"🚀 開始爬取關鍵字: {query} 於看板: r/{subreddit}...")

    while len(all_data) < total_target:
        url = "https://api.pullpush.io/reddit/search/submission/"
        params = {
            'q': query,
            'subreddit': subreddit,
            'size': 100,
            'before': before_time,
            'sort': 'desc'
        }

        try:
            response = requests.get(url, params=params, timeout=15)

            if response.status_code == 200:
                batch = response.json().get('data', [])

                if not batch:
                    print(f"🏁 看板 r/{subreddit} 已無更多符合 [{query}] 的資料。")
                    break

                all_data.extend(batch)

                # 取得這批最後一筆的時間
                last_utc = batch[-1]['created_utc']

                # 💡 關鍵修正：如果這批抓到的數量太少（例如只有1筆），強制將時間往前推12小時
                # 這樣可以避免卡在同一天重複抓取
                if len(batch) < 10:
                    before_time = last_utc - 43200 # 12小時
                else:
                    before_time = last_utc

                # 顯示進度
                current_count = len(all_data)
                last_date = datetime.fromtimestamp(before_time).strftime('%Y-%m-%d')
                print(f"📈 進度: {current_count}/{total_target} 筆 (已處理到: {last_date})")

                time.sleep(1.5) # 稍微休息避免 429 錯誤

            elif response.status_code == 429:
                print("⚠️ 觸發頻率限制 (429)，冷卻 15 秒...")
                time.sleep(15)
            else:
                print(f"❌ 錯誤狀態碼: {response.status_code}")
                break

        except Exception as e:
            print(f"⚠️ 發生意外錯誤: {e}")
            break

    # --- 資料清理區塊 ---
    if not all_data:
        return pd.DataFrame() # 回傳空 DataFrame 避免後續 concat 報錯

    df = pd.DataFrame(all_data)

    # 1. 轉換日期格式
    df['artDate'] = pd.to_datetime(df['created_utc'], unit='s').dt.strftime('%Y-%m-%d')

    # 2. 安全處理網址
    if 'permalink' in df.columns:
        df['url'] = "https://www.reddit.com" + df['permalink']
    else:
        df['url'] = ""

    # 3. 處理內容合併與空值填充
    df['title'] = df['title'].fillna("")
    df['selftext'] = df['selftext'].fillna("")
    df['content'] = df['title'] + " " + df['selftext']

    # 4. 篩選最終需要的欄位
    available_columns = ['artDate', 'content', 'score', 'num_comments', 'url']
    df = df[[col for col in available_columns if col in df.columns]]

    # 5. 去除重複內容
    df = df.drop_duplicates(subset=['content']).reset_index(drop=True)

    return df

# --- 執行區 ---

import pandas as pd
import time

# --- 1. 設定爬取清單 ---
subreddits = ["ChatGPT", "GoogleGemini"]
keywords = ["ChatGPT", "Gemini"]
target_per_combination = 500  # 每個組合想抓的上限

all_results = []

# --- 2. 雙重迴圈執行爬取 ---
for sub in subreddits:
    for kw in keywords:
        print(f"\n🔥 正在執行組合：看板 [r/{sub}] + 關鍵字 [{kw}]")

        # 呼叫你原本定義好的 fetch_pullpush_data 函式
        df_temp = fetch_pullpush_data(kw, sub, total_target=target_per_combination)

        if df_temp is not None and not df_temp.empty:
            # 標註這份資料的來源，方便後續追蹤
            df_temp['source_subreddit'] = sub
            df_temp['search_keyword'] = kw
            all_results.append(df_temp)

        # 每個組合跑完休息一下，避免被 API 封鎖
        time.sleep(5)

# --- 3. 合併並進行「最終分類存檔」 ---
if all_results:
    total_df = pd.concat(all_results, ignore_index=True).drop_duplicates(subset=['content'])
    print(f"\n✅ 總計抓取不重複資料: {len(total_df)} 筆")

    # 全部轉小寫以利精準分類
    content_lower = total_df['content'].str.lower()

    # 分類 A：所有討論到 ChatGPT/GPT 的內容 (不論在哪個看板抓到的)
    df_gpt_final = total_df[content_lower.str.contains('chatgpt|gpt', na=False)]
    df_gpt_final.to_csv("only_chatgpt.csv", index=False, encoding='utf-8-sig')

    # 分類 B：所有討論到 Gemini/Bard 的內容 (不論在哪個看板抓到的)
    df_gemini_final = total_df[content_lower.str.contains('gemini|bard', na=False)]
    df_gemini_final.to_csv("only_gemini.csv", index=False, encoding='utf-8-sig')

    print(f"📁 檔案已分開存儲：")
    print(f"1. only_chatgpt.csv ({len(df_gpt_final)} 筆)")
    print(f"2. only_gemini.csv ({len(df_gemini_final)} 筆)")

else:
    print("❌ 警告：所有組合皆未抓到資料。")


🔥 正在執行組合：看板 [r/ChatGPT] + 關鍵字 [ChatGPT]
🚀 開始爬取關鍵字: ChatGPT 於看板: r/ChatGPT...
📈 進度: 100/500 筆 (已處理到: 2025-05-17)
📈 進度: 200/500 筆 (已處理到: 2025-05-16)
📈 進度: 300/500 筆 (已處理到: 2025-05-15)
📈 進度: 400/500 筆 (已處理到: 2025-05-13)
📈 進度: 500/500 筆 (已處理到: 2025-05-11)

🔥 正在執行組合：看板 [r/ChatGPT] + 關鍵字 [Gemini]
🚀 開始爬取關鍵字: Gemini 於看板: r/ChatGPT...
📈 進度: 100/500 筆 (已處理到: 2025-04-29)
📈 進度: 200/500 筆 (已處理到: 2025-04-20)
📈 進度: 300/500 筆 (已處理到: 2025-04-12)
📈 進度: 400/500 筆 (已處理到: 2025-04-03)
📈 進度: 500/500 筆 (已處理到: 2025-03-22)

🔥 正在執行組合：看板 [r/GoogleGemini] + 關鍵字 [ChatGPT]
🚀 開始爬取關鍵字: ChatGPT 於看板: r/GoogleGemini...
📈 進度: 10/500 筆 (已處理到: 2024-02-21)
📈 進度: 11/500 筆 (已處理到: 2024-02-21)
🏁 看板 r/GoogleGemini 已無更多符合 [ChatGPT] 的資料。

🔥 正在執行組合：看板 [r/GoogleGemini] + 關鍵字 [Gemini]
🚀 開始爬取關鍵字: Gemini 於看板: r/GoogleGemini...
📈 進度: 100/500 筆 (已處理到: 2024-03-02)
📈 進度: 131/500 筆 (已處理到: 2023-05-15)
📈 進度: 132/500 筆 (已處理到: 2023-05-14)
🏁 看板 r/GoogleGemini 已無更多符合 [Gemini] 的資料。

✅ 總計抓取不重複資料: 1086 筆
📁 檔案已分開存儲：
1. only_chatgpt.csv (801 筆)
2. onl